In [ ]:
# =============================================================================
# CELL 1 — Tải dataset
# =============================================================================
# Chạy cell này 1 lần, mất ~10 phút tuỳ tốc độ mạng Kaggle
# Nếu đã tải rồi thì skip

import os
if not os.path.exists("/kaggle/working/goodreads_interactions.csv"):
    os.system("wget -q --show-progress https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv -O /kaggle/working/goodreads_interactions.csv")
    print("Download xong!")
else:
    print("File đã có, skip download.")

In [12]:
# =============================================================================
# CELL 2 — Data Loader (Phase 0)
# =============================================================================

from __future__ import annotations
import json, pickle
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np
import pandas as pd

@dataclass
class DataConfig:
    raw_csv: str = "/kaggle/working/goodreads_interactions.csv"
    out_dir: str = "/kaggle/working/processed"
    n_stages: int = 4
    recommend_threshold: int = 4
    min_user_inter: int = 5
    min_item_inter: int = 5
    require_final_stage: bool = True
    sample_n_users: Optional[int] = None
    sample_seed: int = 1234
    n_val_users: int = 5000
    n_test_users: int = 5000


class GoodreadsChainLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
        Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
        self.user_idx: dict = {}
        self.item_idx: dict = {}
        self.n_user: int = 0
        self.n_item: int = 0
        self.interactions: np.ndarray = None
        self.user_item_map: dict = defaultdict(set)

    def _first_pass_counts(self, chunksize=5_000_000):
        print("[1/4] First pass: counting interactions...")
        user_counts, item_counts = defaultdict(int), defaultdict(int)
        n_rows = 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                  usecols=["user_id","book_id","is_read","rating"],
                                  chunksize=chunksize,
                                  dtype={"user_id":np.int32,"book_id":np.int32,
                                         "is_read":np.int8,"rating":np.int8}):
            for u, c in chunk["user_id"].value_counts().items(): user_counts[u] += c
            for i, c in chunk["book_id"].value_counts().items(): item_counts[i] += c
            n_rows += len(chunk)
            print(f"    ...{n_rows:,} rows", end="\r")
        print(f"\n    total={n_rows:,}, users={len(user_counts):,}, items={len(item_counts):,}")
        return user_counts, item_counts

    def _build_id_maps(self, user_counts, item_counts):
        print("[2/4] Filtering and building id maps...")
        valid_users = {u for u,c in user_counts.items() if c >= self.cfg.min_user_inter}
        valid_items = {i for i,c in item_counts.items() if c >= self.cfg.min_item_inter}
        print(f"    valid users={len(valid_users):,}, items={len(valid_items):,}")
        if self.cfg.sample_n_users is not None:
            rng = np.random.default_rng(self.cfg.sample_seed)
            lst = sorted(valid_users)
            if len(lst) > self.cfg.sample_n_users:
                valid_users = set(rng.choice(lst, size=self.cfg.sample_n_users, replace=False).tolist())
                print(f"    sampled {len(valid_users):,} users")
        self.user_idx = {u:i for i,u in enumerate(sorted(valid_users))}
        self.item_idx = {b:i for i,b in enumerate(sorted(valid_items))}
        self.n_user = len(self.user_idx)
        self.n_item = len(self.item_idx)
        return valid_users, valid_items

    def _second_pass_extract(self, valid_users, valid_items, chunksize=5_000_000):
        print("[3/4] Second pass: extracting chains...")
        rows, n_rows = [], 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                  usecols=["user_id","book_id","is_read","rating"],
                                  chunksize=chunksize,
                                  dtype={"user_id":np.int32,"book_id":np.int32,
                                         "is_read":np.int8,"rating":np.int8}):
            mask = chunk["user_id"].isin(valid_users) & chunk["book_id"].isin(valid_items)
            sub = chunk[mask]
            if len(sub) == 0:
                n_rows += len(chunk)
                continue
            stage = np.zeros(len(sub), dtype=np.int8)
            stage[sub["is_read"].values == 1] = 1
            rated = sub["rating"].values > 0
            stage[rated] = np.maximum(stage[rated], 2)
            recom = sub["rating"].values >= self.cfg.recommend_threshold
            stage[recom] = np.maximum(stage[recom], 3)
            u_idx = sub["user_id"].map(self.user_idx).values
            i_idx = sub["book_id"].map(self.item_idx).values
            rows.append(np.stack([u_idx, i_idx, stage], axis=1))
            n_rows += len(chunk)
            print(f"    ...{n_rows:,} rows, kept {sum(len(r) for r in rows):,}", end="\r")
        self.interactions = np.concatenate(rows).astype(np.int32)
        print(f"\n    kept {len(self.interactions):,} interactions")

    def _post_filter_and_index(self):
        print("[4/4] Post-filter...")
        if self.cfg.require_final_stage:
            last = self.cfg.n_stages - 1
            users_with_last = np.unique(self.interactions[self.interactions[:,2]==last, 0])
            keep = np.isin(self.interactions[:,0], users_with_last)
            self.interactions = self.interactions[keep]
            kept = np.unique(self.interactions[:,0])
            remap = {old:new for new,old in enumerate(kept)}
            self.interactions[:,0] = np.array([remap[u] for u in self.interactions[:,0]], dtype=np.int32)
            inv = {v:k for k,v in self.user_idx.items()}
            self.user_idx = {inv[old]:new for old,new in remap.items()}
            self.n_user = len(kept)
            print(f"    {self.n_user:,} users, {len(self.interactions):,} interactions")
        for u,i,_ in self.interactions:
            self.user_item_map[int(u)].add(int(i))
        stage_counts = np.bincount(self.interactions[:,2], minlength=self.cfg.n_stages)
        print("    stage dist: " + ", ".join(f"stage{l}={c:,}" for l,c in enumerate(stage_counts)))

    def split_train_test(self):
        print("Splitting...")
        rng = np.random.default_rng(self.cfg.sample_seed)
        last = self.cfg.n_stages - 1
        user_final = defaultdict(list)
        for idx,(u,i,s) in enumerate(self.interactions):
            if s == last: user_final[int(u)].append(idx)
        eligible = [u for u,idxs in user_final.items() if len(idxs) >= 3]
        rng.shuffle(eligible)
        n_val  = min(self.cfg.n_val_users,  len(eligible) // 2)
        n_test = min(self.cfg.n_test_users, len(eligible) - n_val)
        val_idx  = [rng.choice(user_final[u]) for u in eligible[:n_val]]
        test_idx = [rng.choice(user_final[u]) for u in eligible[n_val:n_val+n_test]]
        holdout = set(val_idx) | set(test_idx)
        mask = np.ones(len(self.interactions), dtype=bool)
        mask[list(holdout)] = False
        self.data_train = self.interactions[mask]
        self.data_val   = self.interactions[val_idx]
        self.data_test  = self.interactions[test_idx]
        print(f"    train={len(self.data_train):,}, val={len(self.data_val):,}, test={len(self.data_test):,}")

    def build(self):
        uc, ic = self._first_pass_counts()
        vu, vi = self._build_id_maps(uc, ic)
        self._second_pass_extract(vu, vi)
        self._post_filter_and_index()
        self.split_train_test()
        return self

    def save(self):
        out = Path(self.cfg.out_dir)
        np.save(out/"data_train.npy", self.data_train)
        np.save(out/"data_val.npy",   self.data_val)
        np.save(out/"data_test.npy",  self.data_test)
        pickle.dump(self.user_idx,        open(out/"user_idx.pkl","wb"))
        pickle.dump(self.item_idx,        open(out/"item_idx.pkl","wb"))
        pickle.dump(dict(self.user_item_map), open(out/"user_item_map.pkl","wb"))
        json.dump({"n_user":self.n_user,"n_item":self.n_item,"n_stage":self.cfg.n_stages,
                   "config":asdict(self.cfg)},
                  open(out/"meta.json","w"), indent=2)
        print(f"Saved to {out}/")

In [13]:
# =============================================================================
# CELL 3 — Chạy data loader (skip nếu đã có processed data)
# =============================================================================

PROC = Path("/kaggle/working/processed")

if not (PROC / "data_train.npy").exists():
    cfg_data = DataConfig(
        raw_csv        = "/kaggle/working/goodreads_interactions.csv",
        out_dir        = "/kaggle/working/processed",
        sample_n_users = 50000,   # tăng lên None nếu muốn full dataset
        n_val_users    = 5000,
        n_test_users   = 5000,
    )
    loader = GoodreadsChainLoader(cfg_data).build()
    loader.save()
else:
    print("Processed data đã có, skip.")

Processed data đã có, skip.


In [14]:

# =============================================================================
# CELL 4 — Load processed data
# =============================================================================

data_train = np.load(PROC / "data_train.npy")
data_val   = np.load(PROC / "data_val.npy")
data_test  = np.load(PROC / "data_test.npy")
with open(PROC / "user_item_map.pkl", "rb") as f:
    user_item_map = pickle.load(f)
meta = json.loads((PROC / "meta.json").read_text())

print(f"n_user={meta['n_user']:,}, n_item={meta['n_item']:,}, n_stage={meta['n_stage']}")
print(f"train={len(data_train):,}, val={len(data_val):,}, test={len(data_test):,}")


n_user=48,307, n_item=1,567,258, n_stage=4
train=13,695,971, val=5,000, test=5,000


In [15]:


# =============================================================================
# CELL 5 — chainRec Model (Phase 1)
# =============================================================================

import time
from dataclasses import dataclass
from typing import Literal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


@dataclass
class ModelConfig:
    n_user: int
    n_item: int
    n_stage: int       = 4
    embed_dim: int     = 64
    beta: float        = 1.0
    learn_beta: bool   = True
    l2: float          = 0.01
    lr: float          = 0.001
    batch_size: int    = 512
    n_neg: int         = 1
    n_epochs: int      = 50
    patience: int      = 5
    sampler: Literal["uniform", "stagewise"] = "uniform"
    device: str        = "cuda" if torch.cuda.is_available() else "cpu"


class ChainRecModel(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        log_beta_init  = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta:
            self.log_beta = nn.Parameter(log_beta_init)
        else:
            self.register_buffer("log_beta", log_beta_init)
        for emb in [self.user_emb, self.item_emb, self.stage_emb]:
            nn.init.xavier_uniform_(emb.weight)
        for bias in [self.b_user, self.b_item]:
            nn.init.zeros_(bias.weight)

    @property
    def beta(self):
        return torch.clamp(self.log_beta.exp(), min=1.0)

    def _intention_score(self, u, i, l):
        return (self.stage_emb(l) * self.item_emb(i) * self.user_emb(u)).sum(-1)

    def _rectified(self, delta):
        b = self.beta
        return F.softplus(b * delta) / b

    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rectified(self._intention_score(u, i, l_t))
        return bias + acc

    def score_all_stages(self, u, i):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        delta_plus = torch.stack([
            self._rectified(self._intention_score(
                u, i, torch.full((B,), l, dtype=torch.long, device=u.device)))
            for l in range(self.cfg.n_stage)
        ], dim=1)
        suffix_sum = delta_plus.flip(dims=[1]).cumsum(dim=1).flip(dims=[1])
        return bias.unsqueeze(1) + suffix_sum

    def edgewise_terms(self, u, i, l_star):
        L = self.cfg.n_stage
        scores = self.score_all_stages(u, i)
        l_star_c = l_star.clamp(0, L-1)
        s_lstar  = scores.gather(1, l_star_c.unsqueeze(1)).squeeze(1)
        s_next   = scores.gather(1, (l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1)
        s_next   = torch.where(l_star == L-1, torch.full_like(s_next, -1e9), s_next)
        p_lstar  = torch.sigmoid(s_lstar)
        p_next   = torch.sigmoid(s_next)
        delta_p  = self._rectified(self._intention_score(u, i, l_star_c))
        p_cap    = (1.0 - torch.exp(-delta_p)).clamp(min=1e-8)
        return p_lstar, p_next, p_cap


def edgewise_loss(model, u_pos, i_pos, l_pos, u_neg, i_neg, l_neg, l2):
    p_pos, _, _       = model.edgewise_terms(u_pos, i_pos, l_pos)
    _, p_next, p_cap  = model.edgewise_terms(u_neg, i_neg, l_neg)
    loss_pos = -torch.log(p_pos.clamp(min=1e-8)).mean()
    loss_neg = -(torch.log((1-p_next).clamp(min=1e-8)) + torch.log(p_cap)).mean()
    l2_loss  = l2 * (model.user_emb.weight.norm(2)**2 + model.item_emb.weight.norm(2)**2) \
               / (model.cfg.n_user + model.cfg.n_item)
    return loss_pos + loss_neg + l2_loss


class ChainDataset(Dataset):
    def __init__(self, data, user_item_map, n_item, n_neg=1,
                 sampler="uniform", all_data=None):
        self.data = data
        self.user_item_map = user_item_map
        self.n_item = n_item
        self.n_neg  = n_neg
        self.sampler = sampler
        self.rng = np.random.default_rng(42)
        if sampler == "stagewise" and all_data is not None:
            self.user_stage_count = defaultdict(lambda: defaultdict(int))
            for u,i,l in all_data:
                self.user_stage_count[int(u)][int(l)] += 1
        else:
            self.user_stage_count = None

    def _sample_neg(self, u, l_star):
        pos = self.user_item_map.get(u, set())
        for _ in range(30):
            ni = self.rng.integers(0, self.n_item)
            if ni not in pos:
                return int(ni), int(l_star)
        return int(self.rng.integers(0, self.n_item)), int(l_star)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        u, i, l = [int(x) for x in self.data[idx]]
        ni, nl  = self._sample_neg(u, l)
        return {
            "u_pos": torch.tensor(u,  dtype=torch.long),
            "i_pos": torch.tensor(i,  dtype=torch.long),
            "l_pos": torch.tensor(l,  dtype=torch.long),
            "u_neg": torch.tensor(u,  dtype=torch.long),
            "i_neg": torch.tensor(ni, dtype=torch.long),
            "l_neg": torch.tensor(nl, dtype=torch.long),
        }


class Evaluator:
    def __init__(self, model, data_test, user_item_map, n_item,
                 n_neg_eval=500, K_list=None, device="cpu"):
        self.model        = model
        self.data_test    = data_test
        self.user_item_map= user_item_map
        self.n_item       = n_item
        self.n_neg_eval   = n_neg_eval
        self.K_list       = K_list or [10, 20]
        self.device       = device
        self.rng          = np.random.default_rng(999)

    @torch.no_grad()
    def evaluate(self, target_stage=None):
        self.model.eval()
        L = self.model.cfg.n_stage
        if target_stage is None: target_stage = L - 1
        hits = {k: [] for k in self.K_list}
        ndcg = {k: [] for k in self.K_list}
        aucs = []
        for u, i_pos, l_star in self.data_test:
            u, i_pos = int(u), int(i_pos)
            if int(l_star) != target_stage: continue
            pos_items = self.user_item_map.get(u, set())
            negs, attempts = [], 0
            while len(negs) < self.n_neg_eval and attempts < self.n_neg_eval * 5:
                c = self.rng.integers(0, self.n_item)
                if c not in pos_items and c != i_pos: negs.append(c)
                attempts += 1
            all_items = np.array([i_pos] + negs, dtype=np.int64)
            u_t = torch.full((len(all_items),), u, dtype=torch.long, device=self.device)
            i_t = torch.tensor(all_items, dtype=torch.long, device=self.device)
            scores   = self.model.score(u_t, i_t, target_stage).cpu().numpy()
            ranked   = np.argsort(-scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            aucs.append((scores[0] > scores[1:]).mean())
            for k in self.K_list:
                hit = int(pos_rank < k)
                hits[k].append(hit)
                ndcg[k].append((hit / np.log2(pos_rank+2)) / (1/np.log2(2)) if hit else 0.0)
        results = {"AUC": float(np.mean(aucs))}
        for k in self.K_list:
            results[f"Recall@{k}"] = float(np.mean(hits[k]))
            results[f"NDCG@{k}"]   = float(np.mean(ndcg[k]))
        return results


class Trainer:
    def __init__(self, model, cfg, data_train, data_val, data_test, user_item_map):
        self.model     = model.to(cfg.device)
        self.cfg       = cfg
        self.device    = cfg.device
        self.data_val  = data_val
        self.optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
        ds = ChainDataset(data_train, user_item_map, cfg.n_item,
                          n_neg=cfg.n_neg, sampler=cfg.sampler, all_data=data_train)
        self.train_dl  = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                                    num_workers=2, pin_memory=(cfg.device=="cuda"))
        self.evaluator = Evaluator(model, data_test, user_item_map,
                                   cfg.n_item, device=cfg.device)

    def _val_loss(self):
        self.model.eval()
        with torch.no_grad():
            idx   = np.random.permutation(len(self.data_val))[:2000]
            batch = self.data_val[idx]
            u = torch.tensor(batch[:,0], dtype=torch.long, device=self.device)
            i = torch.tensor(batch[:,1], dtype=torch.long, device=self.device)
            l = torch.tensor(batch[:,2], dtype=torch.long, device=self.device)
            p, _, _ = self.model.edgewise_terms(u, i, l)
            return -torch.log(p.clamp(min=1e-8)).mean().item()

    def train(self, save_path="/kaggle/working/chainrec_best.pt"):
        best_val, patience_count, history = float("inf"), 0, []
        print(f"Training on {self.device}")
        print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | "
              f"{'AUC':>6} | {'R@10':>6} | {'NDCG@10':>7} | {'Time':>6}")
        print("-" * 65)
        for epoch in range(1, self.cfg.n_epochs + 1):
            self.model.train()
            t0, total, nb = time.time(), 0.0, 0
            for batch in self.train_dl:
                u_pos = batch["u_pos"].to(self.device)
                i_pos = batch["i_pos"].to(self.device)
                l_pos = batch["l_pos"].to(self.device)
                u_neg = batch["u_neg"].to(self.device)
                i_neg = batch["i_neg"].to(self.device)
                l_neg = batch["l_neg"].to(self.device)
                self.optimizer.zero_grad()
                loss = edgewise_loss(self.model, u_pos, i_pos, l_pos,
                                     u_neg, i_neg, l_neg, self.cfg.l2)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                total += loss.item(); nb += 1
            train_loss = total / nb
            val_loss   = self._val_loss()
            metrics    = self.evaluator.evaluate()
            print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
                  f"{metrics['AUC']:>6.4f} | {metrics['Recall@10']:>6.4f} | "
                  f"{metrics['NDCG@10']:>7.4f} | {time.time()-t0:>5.1f}s")
            history.append({"epoch":epoch,"train_loss":train_loss,
                            "val_loss":val_loss,**metrics})
            if val_loss < best_val:
                best_val, patience_count = val_loss, 0
                torch.save(self.model.state_dict(), save_path)
            else:
                patience_count += 1
                if patience_count >= self.cfg.patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        self.model.load_state_dict(torch.load(save_path, map_location=self.device))
        print("\n=== Final Test Results ===")
        final = self.evaluator.evaluate()
        for k,v in final.items(): print(f"  {k}: {v:.4f}")
        return history, final

In [ ]:
# =============================================================================
# CELL 6 — Train
# =============================================================================

cfg = ModelConfig(
    n_user     = meta["n_user"],
    n_item     = meta["n_item"],
    n_stage    = meta["n_stage"],
    embed_dim  = 16,     # ← giảm từ 64 xuống 16 (~26M params)
    l2         = 0.01,
    lr         = 0.001,
    batch_size = 1024,   # ← tăng batch size để tận dụng GPU
    n_neg      = 1,
    n_epochs   = 50,
    patience   = 5,
    sampler    = "uniform",
)

model = ChainRecModel(cfg)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {cfg.device}")

trainer = Trainer(model, cfg, data_train, data_val, data_test, user_item_map)
history, final_metrics = trainer.train()


Parameters: 27,464,671
Device: cuda
Training on cuda
Epoch | Train Loss |   Val Loss |    AUC |   R@10 | NDCG@10 |   Time
-----------------------------------------------------------------
    1 |     0.9411 |     0.5205 | 0.9395 | 0.7542 |  0.6043 | 837.4s
    2 |     0.4598 |     0.4608 | 0.9468 | 0.7654 |  0.6104 | 810.8s
    3 |     0.4082 |     0.4761 | 0.9487 | 0.7698 |  0.6108 | 817.2s
    4 |     0.3807 |     0.5178 | 0.9496 | 0.7742 |  0.6172 | 883.5s


In [ ]:

# =============================================================================
# CELL 7 — Multi-stage evaluation
# =============================================================================

stage_names = ["shelve", "read", "rate", "recommend"]
print("\n=== Per-Stage Evaluation ===")
for stage in range(meta["n_stage"]):
    stage_data = data_test[data_test[:, 2] == stage]
    if len(stage_data) == 0:
        continue
    ev = Evaluator(model, stage_data, user_item_map,
                   meta["n_item"], device=cfg.device)
    m  = ev.evaluate(target_stage=stage)
    print(f"Stage {stage} ({stage_names[stage]:>10}): "
          f"AUC={m['AUC']:.4f}, R@10={m['Recall@10']:.4f}, NDCG@10={m['NDCG@10']:.4f}")